<a href="https://colab.research.google.com/github/palarunava/machine-learning-courses/blob/main/machine-learning-misc/qwen_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/palarunava/machine-learning-courses.git
!mkdir utils
!mv machine-learning-courses/machine-learning-misc/utils/* utils/
!rm -rf machine-learning-courses

Cloning into 'machine-learning-courses'...
remote: Enumerating objects: 867, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 867 (delta 84), reused 60 (delta 60), pack-reused 748 (from 2)
Receiving objects: 100% (867/867), 9.11 MiB | 16.40 MiB/s, done.
Resolving deltas: 100% (467/467), done.


In [3]:
import logging
import os
from typing import Dict, List, Optional, Union
import unicodedata
from urllib.parse import urlparse

import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from transformers import AutoProcessor, AutoModel
from utils.vision_process import process_vision_info

In [4]:
logger = logging.getLogger(__name__)

MODEL_ID = "Qwen/Qwen3-VL-Embedding-2B"

MAX_LENGTH = 8192
IMAGE_BASE_FACTOR = 16
IMAGE_FACTOR = IMAGE_BASE_FACTOR * 2
FRAME_MAX_PIXELS = 768 * IMAGE_FACTOR * IMAGE_FACTOR
MAX_TOTAL_PIXELS = 10 * FRAME_MAX_PIXELS
MAX_FRAMES = 64
FPS = 1
MIN_PIXELS = 4 * IMAGE_FACTOR * IMAGE_FACTOR
MAX_PIXELS = 1800 * IMAGE_FACTOR * IMAGE_FACTOR

_default_instruction: str = "Represent the user's input."
_total_pixels: int = MAX_TOTAL_PIXELS
_max_frames: int = MAX_FRAMES
_fps: float = FPS
_min_pixels: int = MIN_PIXELS
_max_pixels: int = MAX_PIXELS

In [5]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
model     = AutoModel.from_pretrained(MODEL_ID)
model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/783 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

Qwen3VLModel(
  (visual): Qwen3VLVisionModel(
    (patch_embed): Qwen3VLVisionPatchEmbed(
      (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (pos_embed): Embedding(2304, 1024)
    (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-23): 24 x Qwen3VLVisionBlock(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): Qwen3VLVisionAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (mlp): Qwen3VLVisionMLP(
          (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (act_fn): GELUTanh()
        )
      )
    )
    (merger): Qwen3VLVisionPatchMerger(
      (norm): LayerNorm((1024,), eps=1e-06, ele

In [7]:
def _preprocess_inputs(conversations: List[List[Dict]]) -> Dict[str, torch.Tensor]:
  text = processor.apply_chat_template(
      conversations, add_generation_prompt=True, tokenize=False
  )

  try:
      images, video_inputs, video_kwargs = process_vision_info(
          conversations, image_patch_size=16,
          return_video_metadata=True, return_video_kwargs=True
      )
  except Exception as e:
      logger.error(f"Error in processing vision info: {e}")
      images = None
      video_inputs = None
      video_kwargs = {'do_sample_frames': False}
      text = processor.apply_chat_template(
          [{'role': 'user', 'content': [{'type': 'text', 'text': 'NULL'}]}],
          add_generation_prompt=True, tokenize=False
      )

  if video_inputs is not None:
      videos, video_metadata = zip(*video_inputs)
      videos = list(videos)
      video_metadata = list(video_metadata)
  else:
      videos, video_metadata = None, None

  inputs = processor(
      text=text, images=images, videos=videos, video_metadata=video_metadata, truncation=True,
      max_length=MAX_LENGTH, padding=True, do_resize=False, return_tensors='pt',
      **video_kwargs
  )
  return inputs

In [12]:
def is_image_path(path: str) -> bool:
    image_extensions = {'.jpg', '.jpeg', '.png', '.gif', '.bmp', '.webp', '.tiff', '.svg'}

    if path.startswith(('http://', 'https://')):
        # Parse URL to remove query parameters
        parsed_url = urlparse(path)
        clean_path = parsed_url.path
    else:
        clean_path = path

    # Check file extension
    _, ext = os.path.splitext(clean_path.lower())
    return ext in image_extensions

In [13]:
def is_video_input(video) -> bool:
    if isinstance(video, str):
        return True

    if isinstance(video, list) and len(video) > 0:
        # Check first element to determine the type
        first_elem = video[0]

        if isinstance(first_elem, Image.Image):
            return True

        if isinstance(first_elem, str):
            return is_image_path(first_elem)

    return False

In [14]:
def sample_frames(frames: List[Union[str, Image.Image]], max_segments: int) -> List[Union[str, Image.Image]]:
    duration = len(frames)
    if duration <= max_segments:
        return frames

    frame_id_array = np.linspace(0, duration - 1, max_segments, dtype=int)
    frame_id_list = frame_id_array.tolist()
    sampled_frames = [ frames[frame_idx] for frame_idx in frame_id_list ]
    return sampled_frames

In [15]:
def format_model_input(
    text: Optional[Union[List[str], str]] = None,
    image: Optional[Union[List[Union[str, Image.Image]], str, Image.Image]] = None,
    video: Optional[Union[List[Union[str, List[Union[str, Image.Image]]]], str, List[Union[str, Image.Image]]]] = None,
    instruction: Optional[str] = None,
    fps: Optional[float] = None,
    max_frames: Optional[int] = None
) -> List[Dict]:

    # Ensure instruction ends with punctuation
    if instruction:
        instruction = instruction.strip()
        if instruction and not unicodedata.category(instruction[-1]).startswith('P'):
            instruction = instruction + '.'

    # Initialize conversation with system prompts
    content = []
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": instruction or _default_instruction}]},
        {"role": "user", "content": content}
    ]

    # Normalize text input to list
    if text is None:
        texts = []
    elif isinstance(text, str):
        texts = [text]
    else:
        texts = text

    # Normalize image input to list
    if image is None:
        images = []
    elif not isinstance(image, list):
        images = [image]
    else:
        images = image

    # Normalize video input to list
    if video is None:
        videos = []
    elif is_video_input(video):
        videos = [video]
    else:
        # Assume it's a list of videos
        videos = video

    # Add text, image, or video content to conversation
    if not texts and not images and not videos:
        content.append({'type': 'text', 'text': "NULL"})
        return conversation

    # Process each video
    for vid in videos:
        video_content = None
        video_kwargs = {'total_pixels': _total_pixels}

        if isinstance(vid, list):
            # Video as frame sequence
            video_content = vid
            if _max_frames is not None:
                video_content = sample_frames(video_content, _max_frames)
            video_content = [
                ('file://' + ele if isinstance(ele, str) else ele)
                for ele in video_content
            ]
        elif isinstance(vid, str):
            # Video as file path
            video_content = vid if vid.startswith(('http://', 'https://')) else 'file://' + vid
            video_kwargs = {'fps': fps or _fps, 'max_frames': max_frames or _max_frames}
        else:
            raise TypeError(f"Unrecognized video type: {type(vid)}")

        # Add video input to content
        if video_content:
            content.append({
                'type': 'video',
                'video': video_content,
                **video_kwargs
            })

    # Process each image
    for img in images:
        image_content = None

        if isinstance(img, Image.Image):
            image_content = img
        elif isinstance(img, str):
            image_content = img if img.startswith(('http://', 'https://')) else 'file://' + img
        else:
            raise TypeError(f"Unrecognized image type: {type(img)}")

        # Add image input to content
        if image_content:
            content.append({
                'type': 'image',
                'image': image_content,
                "min_pixels": _min_pixels,
                "max_pixels": _max_pixels
            })

    # Process each text
    for txt in texts:
        content.append({'type': 'text', 'text': txt})

    return conversation

In [16]:
def _pooling_last(hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    flipped_tensor = attention_mask.flip(dims=[1])
    last_one_positions = flipped_tensor.argmax(dim=1)
    col = attention_mask.shape[1] - last_one_positions - 1
    row = torch.arange(hidden_state.shape[0], device=hidden_state.device)
    return hidden_state[row, col]

In [17]:
inputs = [
    {
        "text": "A woman playing with her dog on a beach at sunset.",
        "instruction": "Retrieve images or text relevant to the user's query.",
    },
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust."
    },
    {
        "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
    },
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust.",
        "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
    }
]

conversations = [format_model_input(
    text=ele.get('text'),
    image=ele.get('image'),
    video=ele.get('video'),
    instruction=ele.get('instruction'),
    fps=ele.get('fps'),
    max_frames=ele.get('max_frames')
) for ele in inputs]

processed_inputs = _preprocess_inputs(conversations)
processed_inputs = {k: v.to(model.device) for k, v in processed_inputs.items()}

In [ ]:
outputs = model(**processed_inputs)
outputs = {
    'last_hidden_state': outputs.last_hidden_state,
    'attention_mask': inputs.get('attention_mask')
}
embeddings = F.normalize(embeddings, p=2, dim=-1)

In [ ]:
print("Embeddings shape:", embeddings.shape)
print("Embeddings:", embeddings)